# P1 — Inventaire des tables · Fuites

**Sujet 05 — Santander Satisfaction**  
**Porteur P1** · Enchaînement : 1.1 → 1.8

| Section | Titre | Question clé |
|---|---|---|
| 1.1 | Inventorier les tables | Qu'est-ce qu'on a comme données ? |
| 1.8 | Fuites | Y a-t-il des variables qui trichent ? |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

In [ ]:
# Localisation des données
FICHIERS_ATTENDUS = ['train.csv', 'a_decider.csv']
SUJET = "Sujet_05_Santander_Satisfaction"

def trouver_donnees(fichiers=FICHIERS_ATTENDUS, sujet=SUJET):
    pistes = []
    if os.environ.get("DONNEES_PROJET"):
        pistes.append(Path(os.environ["DONNEES_PROJET"]))
    ici = Path.cwd().resolve()
    pistes += [ici, ici / "donnees", ici / "data", ici / sujet]
    pistes += list(ici.parents)[:4]
    pistes += [Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")]
    essais = []
    for racine in pistes:
        if not racine.exists():
            continue
        candidates = [racine, racine / sujet]
        if racine.name in ("input", "MyDrive"):
            candidates += [d for d in racine.iterdir() if d.is_dir()]
        for c in candidates:
            essais.append(str(c))
            if all((c / f).is_file() for f in fichiers):
                return c
    raise FileNotFoundError(
        "Données introuvables.\n"
        "Placez train.csv et a_decider.csv dans le même dossier que ce carnet.\n"
        "Chemins essayés : " + str(essais[:12])
    )

DOSSIER = trouver_donnees()
print("Données trouvées :", DOSSIER)

---
## 1.1 — Inventorier les tables

On recense toutes les tables disponibles, leurs dimensions, et on inspecte leur contenu global.

In [ ]:
# Chargement des deux tables
train = pd.read_csv(DOSSIER / "train.csv")
a_decider = pd.read_csv(DOSSIER / "a_decider.csv")

print("=== INVENTAIRE DES TABLES ===")
print(f"\ntrain.csv     : {train.shape[0]:,} lignes × {train.shape[1]} colonnes")
print(f"a_decider.csv : {a_decider.shape[0]:,} lignes × {a_decider.shape[1]} colonnes")

In [ ]:
# Colonnes présentes dans train mais absentes dans a_decider (et inversement)
cols_train = set(train.columns)
cols_decider = set(a_decider.columns)

only_train = cols_train - cols_decider
only_decider = cols_decider - cols_train
communes = cols_train & cols_decider

print(f"Colonnes communes           : {len(communes)}")
print(f"Colonnes uniquement train   : {only_train}")
print(f"Colonnes uniquement decider : {only_decider}")

In [ ]:
# Familles de colonnes (préfixe)
prefixes = {}
for col in train.columns:
    if col in ('ID', 'TARGET'):
        prefixes[col] = prefixes.get(col, 0) + 1
        continue
    pref = col.split('_')[0]
    prefixes[pref] = prefixes.get(pref, 0) + 1

print("Familles de colonnes :")
for pref, nb in sorted(prefixes.items(), key=lambda x: -x[1]):
    print(f"  {pref:20s} : {nb} colonnes")

In [ ]:
# Aperçu des premières lignes
print("Aperçu de train.csv (5 premières lignes, 8 premières colonnes) :")
display(train.iloc[:5, :8])

In [ ]:
# Statistiques globales
print("Statistiques descriptives (10 premières colonnes numériques) :")
display(train.describe().iloc[:, :10].round(2))

In [ ]:
# Colonnes constantes (variance nulle)
constantes = [col for col in train.columns if train[col].nunique() == 1]
print(f"Colonnes constantes (variance nulle) : {len(constantes)}")
if constantes:
    print("  Exemples :", constantes[:10])

In [ ]:
print("=== BILAN 1.1 — Inventaire ===")
print(f"  Tables         : train.csv ({train.shape[0]:,} lignes) + a_decider.csv ({a_decider.shape[0]:,} lignes)")
print(f"  Colonnes train : {train.shape[1]} dont {len(constantes)} constantes")
print(f"  Colonne cible  : TARGET (absente dans a_decider, à prédire)")
print(f"  Familles       : saldo_, imp_, num_, ind_, delta_, var*")

---
## 1.8 — Fuites (Data Leakage)

Une **fuite** est une variable qui contient de l'information sur la cible future, mais qui ne serait **pas disponible au moment réel de la décision**. L'inclure dans le modèle donne des scores artificiellement bons qui ne se confirment pas en production.

On cherche ici des variables **suspectes** : celles dont la distribution diffère fortement entre `TARGET=0` et `TARGET=1`.

In [ ]:
CIBLE = "TARGET"
satisfaits = train[train[CIBLE] == 0]
mecontents = train[train[CIBLE] == 1]

print(f"Clients satisfaits (TARGET=0) : {len(satisfaits):,}")
print(f"Clients mécontents (TARGET=1) : {len(mecontents):,}")

In [ ]:
# Détection des variables suspectes de fuite
# Critère : proportion de valeurs nulles très différente selon la cible
features = [c for c in train.columns if c not in ('ID', CIBLE)]
suspects = []

for col in features:
    taux_zero_sat = (satisfaits[col] == 0).mean()
    taux_zero_mec = (mecontents[col] == 0).mean()
    diff = abs(taux_zero_sat - taux_zero_mec)
    if diff > 0.3:  # seuil : 30% d'écart
        suspects.append({
            'colonne': col,
            'taux_zero_satisfaits': round(taux_zero_sat, 3),
            'taux_zero_mecontents': round(taux_zero_mec, 3),
            'ecart': round(diff, 3)
        })

suspects_df = pd.DataFrame(suspects).sort_values('ecart', ascending=False)
print(f"Variables suspectes de fuite (écart > 30%) : {len(suspects_df)}")
display(suspects_df.head(20))

In [ ]:
# Visualisation des variables les plus suspectes
if len(suspects_df) > 0:
    top_suspects = suspects_df.head(5)['colonne'].tolist()
    fig, axes = plt.subplots(1, min(5, len(top_suspects)), figsize=(14, 4))
    if len(top_suspects) == 1:
        axes = [axes]
    for ax, col in zip(axes, top_suspects):
        vals_sat = satisfaits[col].clip(upper=satisfaits[col].quantile(0.99))
        vals_mec = mecontents[col].clip(upper=mecontents[col].quantile(0.99))
        ax.hist(vals_sat, bins=30, alpha=0.5, label='Satisfaits', color='steelblue', density=True)
        ax.hist(vals_mec, bins=30, alpha=0.5, label='Mécontents', color='crimson', density=True)
        ax.set_title(col[:25], fontsize=8)
        ax.legend(fontsize=7)
    plt.suptitle("Variables suspectes de fuite — distribution par cible")
    plt.tight_layout()
    plt.savefig("../sorties/fuites.png", dpi=100)
    plt.show()
    print("Figure sauvegardée dans sorties/fuites.png")

In [ ]:
print("=== BILAN 1.8 — Fuites ===")
print(f"  Variables analysées          : {len(features)}")
print(f"  Variables suspectes (>30%d) : {len(suspects_df)}")
print()
print("  Rappel : une fuite est une variable disponible AVANT la plainte")
print("  mais dont la valeur change avec la plainte → à investiguer")
if len(suspects_df) > 0:
    print(f"  Top suspect : {suspects_df.iloc[0]['colonne']} (écart={suspects_df.iloc[0]['ecart']})")